# AKRA memory & cost budget

The inverse problem is $y = Ax + n$, with the generalized-least-squares solution

$$\hat{x} = (A^\top N^{-1} A)^{-1}\, A^\top N^{-1} y .$$

Working in spherical-harmonic space up to $\ell_{\max}$, the unknown $x$ has
$N_{\rm modes} = (\ell_{\max}+1)^2$ components.

- **AKRA 2.0** forms the normal matrix $A^\top N^{-1} A$ explicitly. It is
  $N_{\rm modes}\times N_{\rm modes}$, so storage scales as $N_{\rm modes}^2$ — prohibitive at survey resolution.
- **AKRA 3.0** never forms that matrix. $A^\top N^{-1} A$ is applied as a *linear operator*
  and the system is solved with **conjugate gradient (CG)**, which only needs a handful of
  mode-length vectors — storage scales as $N_{\rm modes}$.

This notebook quantifies that difference.

In [2]:
import numpy as np
import healpy as hp

BYTES = 8  # float64 storage per element


def fmt_mem(nbytes):
    """Human-readable memory string."""
    for unit, scale in [("PB", 1e15), ("TB", 1e12), ("GB", 1e9), ("MB", 1e6)]:
        if nbytes >= scale:
            return f"{nbytes / scale:.2f} {unit}"
    return f"{nbytes / 1e3:.2f} KB"


def n_modes(ell_max):
    """Number of spherical-harmonic modes a_lm for 0 <= ell <= ell_max."""
    return (ell_max + 1) ** 2

In [3]:
# =====================================================================
#  Helper: FLOP counts
# =====================================================================
def flops_cholesky(n):
    """Cholesky decomposition of n×n SPD matrix."""
    return n**3 / 3.0

def flops_svd(n):
    """Full SVD of n×n dense matrix (LAPACK dgesdd)."""
    return 22.0 * n**3

## 1. AKRA 2.0 — store the dense normal matrix $A^\top N^{-1} A$

The matrix is $(N_{\rm modes}\times N_{\rm modes})$; storage $= N_{\rm modes}^2 \times 8$ bytes.

In [4]:
print("AKRA 2.0  (store & invert dense A^T N^-1 A on the full sphere)")
print(f"{'nside':>6} {'ell_max':>8} {'N_modes':>12} {'dense matrix':>14}")
for nside in [128, 2048, 4096]:
    ell_max = 2 * nside
    Nm = n_modes(ell_max)
    mem_dense = Nm ** 2 * BYTES
    print(f"{nside:>6} {ell_max:>8} {Nm:>12} {fmt_mem(mem_dense):>14}")


AKRA 2.0  (store & invert dense A^T N^-1 A on the full sphere)
 nside  ell_max      N_modes   dense matrix
   128      256        66049       34.90 GB
  2048     4096     16785409        2.25 PB
  4096     8192     67125249       36.05 PB


In [13]:
BLAS_1CORE  = 15      # GFLOP/s per core (sustained, double precision)
SCALING_EFF = 0.70    # parallel efficiency
ncpu        = 72
blas_GFLOPS = BLAS_1CORE * ncpu * SCALING_EFF      # GFLOP/s
PEAK        = blas_GFLOPS * 1e9                     # FLOP/s

for nside in [128, 2048, 4096]:
    ell_max = 2 * nside
    N = n_modes(ell_max)
    flop_form = 2*N**3
    flop_inv  = flops_cholesky(N)
    total_flop = flop_form + flop_inv
    time_sec = total_flop / (PEAK)
    time_yr = time_sec / (86400 * 365.25)
    if nside == 128:
        t_str = f"{time_sec/60:.1f} min"
    else:
        t_str = f"{time_sec/(86400*365.25):.1e} yr"

    print(f"nside={nside:>4}, N={N:>12,}, time={t_str}")

nside= 128, N=      66,049, time=14.8 min
nside=2048, N=  16,785,409, time=4.6e+02 yr
nside=4096, N=  67,125,249, time=3.0e+04 yr


## 2. AKRA 3.0 — matrix-free operator + conjugate gradient

CG only keeps a few mode-length work vectors ($x$, residual $r$, search direction $p$,
operator product $Ap$). Storage $\approx (\text{a few}) \times N_{\rm modes} \times 8$ bytes —
no $N_{\rm modes}^2$ matrix is ever allocated.

In [14]:
N_CG_VEC = 4  # x, r, p, Ap

print("AKRA 3.0  (matrix-free linear operator + conjugate gradient)")
print(f"{'nside':>6} {'ell_max':>8} {'N_modes':>12} {'CG vectors':>12} {'reduction':>12}")
for nside in [128, 2048, 4096]:
    ell_max = 2 * nside
    Nm = n_modes(ell_max)
    mem_dense = Nm ** 2 * BYTES
    mem_op = N_CG_VEC * Nm * BYTES
    print(f"{nside:>6} {ell_max:>8} {Nm:>12} {fmt_mem(mem_op):>12} {mem_dense / mem_op:>10.1e}x")

AKRA 3.0  (matrix-free linear operator + conjugate gradient)
 nside  ell_max      N_modes   CG vectors    reduction
   128      256        66049      2.11 MB    1.7e+04x
  2048     4096     16785409    537.13 MB    4.2e+06x
  4096     8192     67125249      2.15 GB    1.7e+07x


At DES Y3 resolution (`nside = 2048`, `ell_max = 4096`) the dense matrix needs **~2 PB**,
while the operator form needs **< 1 GB** — a reduction of more than six orders of magnitude.
This is what makes AKRA 3.0 feasible on a single machine.

In [37]:
SCALING_EFF = 0.50    # parallel efficiency
blas_GFLOPS = BLAS_1CORE * ncpu * SCALING_EFF      # GFLOP/s
PEAK        = blas_GFLOPS * 1e9         

In [38]:
C_SHT = 20.0  
### CSHT​≈20 is the expected order for a complex spin-2 SHT. A pure-Python or non-vectorized recurrence instead of a compiled kernel. That inflates a wall-clock constant by 10–100×. theoretically is 1.

def n_pix(nside):
    return 12 * nside ** 2

def flops_sht(nside, ell_max):
    """One spin-0 SHT (~ O(ell_max^3)); Legendre term + FFT term."""
    legendre = C_SHT * n_pix(nside) * ell_max
    fft      = n_pix(nside) * 5.0                       # negligible
    return legendre + fft
 
def flops_cg_iter(nside, ell_max):
    """One CG iteration = one application of M = A^T N^-1 A (2 SHTs)
    plus cheap pixel- and mode-space pointwise work."""
    op  = 2.0 * flops_sht(nside, ell_max)               # forward + backward SHT
    op += 5.0 * n_pix(nside)                             # N^-1, mask
    op += 5.0 * n_modes(ell_max)                         # ell-dependent A factors
    vec = 5.0 * 2.0 * n_modes(ell_max)                   # CG axpy/dot bookkeeping
    return op + vec
 
def time_akra3(nside, n_iter):
    ell_max = 2 * nside
    return n_iter * flops_cg_iter(nside, ell_max) / PEAK

N_ITER = [10, 80, 200]
for nside, niter in zip([128, 2048, 4096], N_ITER):
    ell_max = 2 * nside
    Nm      = n_modes(ell_max)
    total_sec      = time_akra3(nside, niter)
    if total_sec < 60:
        t_str = f"{total_sec:.1f} sec"
    elif total_sec < 3600:
        t_str = f"{total_sec/60:.1f} min"
    else:
        t_str = f"{total_sec/3600:.1f} hr"
    print(f"nside={nside:>4}, N={N:>12,}, time={t_str}")


nside= 128, N=      22,500, time=0.0 sec
nside=2048, N=      22,500, time=20.4 min
nside=4096, N=      22,500, time=6.8 hr


## 3. Scale-split mitigation (an AKRA 2.0 work-around)

Before the operator approach, one could keep AKRA 2.0 tractable by *splitting scales*:
solve the low-$\ell$ part on the sphere (small dense matrix) and the high-$\ell$ part on
flat-sky patches (small dense matrix per patch). Even so, the total storage for the DES Y3
footprint is hundreds of GB, and the scheme is awkward — AKRA 3.0 removes the need for it.

In [24]:
nside = 2048
resol_arcmin = hp.nside2resol(nside, arcmin=True)   # pixel scale at nside=2048
print(f"nside={nside}, pixel scale = {resol_arcmin:.2f} arcmin")

# low-ell sphere block
ell_split = 180
Nm_lo = n_modes(ell_split)
mem_lo = Nm_lo ** 2 * BYTES

# high-ell flat-sky patches
grid_spacing_deg = resol_arcmin / 60.0      # pixel size [deg]
N_g = 150                                   # grid points per patch side (fixed; avoids float round-off)
patch_deg = N_g * grid_spacing_deg          # patch angular size [deg]
Nm_patch = N_g ** 2                         # real-space modes per patch
mem_patch = Nm_patch ** 2 * BYTES

survey_area = 4143.0                         # DES Y3 footprint [deg^2]
N_patch = int(np.ceil(survey_area / patch_deg ** 2))

print("\nScale-split (AKRA 2.0 mitigation, DES Y3)")
print(f"  low-ell sphere : ell<{ell_split}, N_modes={Nm_lo}, dense = {fmt_mem(mem_lo)}")
print(f"  flat patch     : {patch_deg:.2f} deg/side, N_g={N_g}, dense/patch = {fmt_mem(mem_patch)}")
print(f"  N_patch        : {N_patch}")
print(f"  peak (1 block at a time) : {fmt_mem(max(mem_lo, mem_patch))}")
print(f"  total (all blocks stored): {fmt_mem(mem_lo + N_patch * mem_patch)}")

nside=2048, pixel scale = 1.72 arcmin

Scale-split (AKRA 2.0 mitigation, DES Y3)
  low-ell sphere : ell<180, N_modes=32761, dense = 8.59 GB
  flat patch     : 4.29 deg/side, N_g=150, dense/patch = 4.05 GB
  N_patch        : 225
  peak (1 block at a time) : 8.59 GB
  total (all blocks stored): 919.84 GB


In [32]:
### for nside=128, we adopt the flat-sky approximation over a localized $10^\circ \times 10^\circ$ patch at a HEALPix resolution of $N_{\text{side}}=128$. Given this specific field of view, the geometric distortions introduced by projecting the spherical sky onto a 2D Cartesian grid are negligible. This configuration allows us to safely apply flat-sky assumptions, effectively reducing computational complexity while maintaining the accuracy of the local spatial statistics."

Npatch = [1, 225, 900]
Npix = []
for nside, npatch in zip([128, 2048, 4096], Npatch):
    time_total = 15 * npatch ## min
    if nside == 128:
        t_str = f"{time_total:.1f} min"
    else:
        t_str = f"{time_total/(12*60):.1e} day"

    print(f"nside={nside:>4}, time={t_str}")

nside= 128, time=15.0 min
nside=2048, time=4.7e+00 day
nside=4096, time=1.9e+01 day
